# Institutional vs retail seasonality in cryptocurrency momentum

**Hypothesis:** cross-sectional momentum is stronger on weekdays, when institutions trade,
and weaker or reversed on weekends, when retail dominates.

**Design**
* Daily cross-sectional momentum (7-day trailing average return, rank-demeaned, normalized,
  1-day holding, 20 bps costs).
* Results are split by day of week. The return is attributed to the day it was earned.
* The weekday/weekend difference is tested with a regression of net returns on a weekend
  dummy, using Newey-West standard errors (7 lags, for daily autocorrelation).

**Caveats**
* Daily momentum rebalances almost every day, so costs are large. Gross and net results are
  both shown; the net result is the one that matters.
* Seven day-of-week tests are run. Treat any single p-value as descriptive.

Every number is printed by code.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import statsmodels.api as sm

from cryptolib import *  # noqa: F401,F403

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

UNIVERSE = ["BTCUSDT", "ETHUSDT", "ADAUSDT", "BNBUSDT", "XRPUSDT", "DOTUSDT", "MATICUSDT"]
START = "2020-01-01"
END = None
FORMATION_DAYS = 7          # fixed a priori
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

## 1. Data and daily backtest

In [ ]:
close_d, _ = load_universe(UNIVERSE, START, END)
px = common_window(close_d)
ret = px.pct_change().dropna()
print(f"Common daily window: {px.index[0].date()} -> {px.index[-1].date()}  ({len(ret)} days)")

bt = momentum_backtest(ret, formation=FORMATION_DAYS)
print(f"\nDaily strategy (all days, {FORMATION_DAYS}-day formation):")
print(metrics_table({"Gross": bt["gross"], "Net (20 bps)": bt["net"]}, PPY_DAILY).to_string())
print()
print(cost_summary(bt, PPY_DAILY).to_string())

## 2. Performance by day of week
Sharpe ratios for a single weekday are computed on one observation per week, so they are
annualized with 52 periods.

In [ ]:
def sharpe_52(x):
    x = pd.Series(x).dropna()
    sd = x.std(ddof=1)
    return x.mean() / sd * np.sqrt(52) if sd > 0 else np.nan


dow = bt.index.dayofweek
rows = []
for d in range(7):
    s = bt.loc[dow == d]
    rows.append({"Day": DAY_NAMES[d],
                 "N": len(s),
                 "Gross mean (bps/day)": s["gross"].mean() * 1e4,
                 "Net mean (bps/day)": s["net"].mean() * 1e4,
                 "Gross Sharpe": sharpe_52(s["gross"]),
                 "Net Sharpe": sharpe_52(s["net"])})
dow_table = pd.DataFrame(rows).set_index("Day")
print(dow_table.to_string())

## 3. Weekday vs weekend test
Regression of daily net return on a weekend dummy (Sat/Sun = 1). The coefficient is the
weekend minus weekday difference in mean daily return. HAC standard errors, 7 lags.

In [ ]:
weekend = pd.Series((dow >= 5).astype(float), index=bt.index, name="weekend")
X = sm.add_constant(weekend)
for label, col in [("Net", "net"), ("Gross", "gross")]:
    m = sm.OLS(bt[col], X).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
    print(f"{label}: weekday mean = {m.params['const'] * 1e4:+.2f} bps/day   "
          f"weekend - weekday = {m.params['weekend'] * 1e4:+.2f} bps/day   "
          f"t = {m.tvalues['weekend']:.2f}   p = {m.pvalues['weekend']:.4f}")

wd = bt.loc[dow < 5]
we = bt.loc[dow >= 5]
print(f"\nNet Sharpe (daily, 365 periods): weekdays {perf_metrics(wd['net'], PPY_DAILY)['Sharpe']:.2f}"
      f"   weekends {perf_metrics(we['net'], PPY_DAILY)['Sharpe']:.2f}")
print(f"Weekdays: {len(wd)} days   Weekends: {len(we)} days")

## 4. Robustness: by calendar year
A weekend effect that only appears in one year is not a stable seasonal pattern.

In [ ]:
year_rows = {}
for y, g in bt.groupby(bt.index.year):
    gw = g.loc[g.index.dayofweek < 5, "net"]
    ge = g.loc[g.index.dayofweek >= 5, "net"]
    year_rows[y] = {"Weekday net (bps/day)": gw.mean() * 1e4,
                    "Weekend net (bps/day)": ge.mean() * 1e4,
                    "Weekend - weekday (bps/day)": (ge.mean() - gw.mean()) * 1e4,
                    "Days": len(g)}
print(pd.DataFrame(year_rows).T.to_string())

## 5. Chart

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(DAY_NAMES, dow_table["Net Sharpe"].values,
       color=["#1f77b4"] * 5 + ["#ff7f0e"] * 2, alpha=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Net Sharpe of daily momentum by day of week (blue = weekday, orange = weekend)")
ax.set_ylabel("Annualized Sharpe")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("seasonality_dow_sharpe.png", dpi=150)
plt.show()

## 6. How to read the results
* The hypothesis is supported only if the weekend - weekday difference in **net** return is
  significant (section 3) and has the same sign in most years (section 4).
* If the gross difference is significant but the net one is not, the effect is too small
  to trade after costs.
* A null result here is still a valid result: it says the day-of-week pattern does not
  survive trading costs or is not stable across years.

In [ ]:
print("Summary (all numbers computed above):")
diff_net = sm.OLS(bt["net"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
print(f"  Weekend - weekday net difference: {diff_net.params['weekend'] * 1e4:+.2f} bps/day "
      f"(p = {diff_net.pvalues['weekend']:.4f})")
print(f"  Years with weekend > weekday (net): "
      f"{sum(v['Weekend - weekday (bps/day)'] > 0 for v in year_rows.values())} of {len(year_rows)}")